In [76]:
from pytanga.algebra import MV
from pytanga.geometry import Geometry, Point, Translator, Direction, Sphere
from pytanga.basis import BasisN3

N3 = BasisN3()
geo = Geometry(N3)


In [84]:
p = geo.create(Point(1,0,0))

# reflection in plane normal to e1 passing through origin
op = N3("e1")
q = op.vp(p)
q.show("q")
print(geo.which_entity(q))

print(type(op[N3.E1]))

q: e1 - 0.5 einf - eo

Point(-1.00, -0.00, -0.00)
<class 'float'>


In [ ]:
p = geo.create(Point(0,1,0))

# reflection on point at infinity, maps to infinity
op = N3.e1 ^ N3.einf
q = op.vp(p)
q.show("q")
print(geo.which_entity(q))

q: 2 einf

Dir(0.00, 0.00, 0.00)


In [26]:
a1 = geo.create(Point(0,1,0))
a2 = geo.create(Point(1,1,0))
# reflection line, same as line
op = a1 ^ a2 ^ N3.einf
op.show("op")
print(geo.which_entity(op))

p = geo.create(Point(1,0,1))
print(geo.which_entity(p))
q = op.vp(p)
q.show("q")
print(geo.which_entity(q))

# Dual of line
op_d = op.dual()
q = op_d.vp(p)
print(geo.which_entity(q))


op: - e12∧einf + e1∧einf∧eo

Line(org=Point(0.00, 1.00, 0.00), dir=Dir(1.00, 0.00, 0.00))
Point(1.00, 0.00, 1.00)


q: - e1 - 2 e2 + e3 - 3 einf - eo

Point(1.00, 2.00, -1.00)
Point(1.00, 2.00, -1.00)


In [27]:
a1 = geo.create(Point(0,1,0))
a2 = geo.create(Point(1,1,0))
a3 = geo.create(Point(0,0,0))
# reflection plane same as plane
op = a1 ^ a2 ^ a3 ^ N3.einf
op.show("op")
print(geo.which_entity(op))

p = geo.create(Point(1,0,1))
print(geo.which_entity(p))
q = op.vp(p)
q.show("q")
print(geo.which_entity(q))

# Dual of plane
op_d = op.dual()
q = op_d.vp(p)
print(geo.which_entity(q))

op: e12∧einf∧eo

Plane(pt=Point(-0.00, -0.00, -0.00), n=Dir(-0.00, -0.00, -1.00))
Point(1.00, 0.00, 1.00)


q: e1 - e3 + einf + eo

Point(1.00, 0.00, -1.00)
Point(1.00, -0.00, -1.00)


In [ ]:
a1 = geo.create(Point(0,1,0))
# reflection in origin point
E = N3.einf ^ N3.eo
trans = geo.create(Translator(Direction(1,0,0)))

# operator is reflection in point (1,0,0), as T E ~T
op = trans.vp(E)
op.show("op")
print(geo.which_entity(op))

op = geo.create(Point(1,0,0)) ^ N3.einf
op.show("op as hpoint")

p = geo.create(Point(1,2,3))
print(geo.which_entity(p))
q = op.vp(p)
q.show("q")
print(geo.which_entity(q))

# Dual of plane
op_d = op.dual()
q = op_d.vp(p)
print(geo.which_entity(q))

# analysis of operator
# if abs(s) > 0 then we have the E part.
s = op.ip(E)
print(f"s: {s}")
# origin of reflection
t_e = -op.ip(N3.eo).op(N3.eo).ip(N3.einf)
t_e.show("t_e")

op: - e1∧einf + einf∧eo

HPoint(Point(1.00, 0.00, 0.00), w=-1.00)


op as hpoint: e1∧einf - einf∧eo

Point(1.00, 2.00, 3.00)


q: e1 - 2 e2 - 3 e3 + 7 einf + eo

Point(1.00, -2.00, -3.00)
Point(1.00, -2.00, -3.00)
s: -1


t_e: - e1

In [ ]:
def get_op_type(op) -> tuple[str, MV]:
    op_type: str = "unkown"

    is_blade = len(op.grades) == 1
    v_scale, v_list = op.blade_factorize_versor()
    v_grade = len(v_list)
    print(f"Versor grade: {v_grade}")

    if not is_blade:
        print("Need general versor analysis")
        return "tbd"

    if v_grade >= 3:
        op = op.dual()
        v_grade = 5 - v_grade

    op.show("op")
    print(f"New versor grade: {v_grade}")
    
    if v_grade == 1:
        # Test for inversion
        if abs(op.ip(N3.einf).scalar) > 1e-15:
            # op is of the form of an IPNS Sphere
            op_type = "inversion"
        # Test for plane
        elif abs(op[N3.E1]) > 1e-15 or abs(op[N3.E2]) > 1e-15 or abs(op[N3.E3]) > 1e-15:
            # op is of the form of an IPNS plane
            op_type = "reflection plane"
    elif v_grade == 2:
        # Test for reflection in a point
        if abs(op.ip(N3.einf^N3.eo).scalar) > 1e-15:
            # op is of the form of an HPoint (Point ^ einf)
            op_type = "ref_in_point"
        # Test for reflection in a line
        elif abs(op[N3.E12]) > 1e-15 or abs(op[N3.E23]) > 1e-15 or abs(op[N3.E13]) > 1e-15:
            # op is of the form of an IPNS line
            op_type = "ref_in_line"

    return op_type, op

In [86]:
# Inversion in a sphere at a point
s1 = geo.create(Sphere(Point(1,2,3), 1))
op = s1
op.show("s1")
op_d = op.dual()
op_d.show("op_d")

p = geo.create(Point(0,0,0))
q1 = op.vp(p)
print(geo.which_entity(q1))
q2 = op_d.vp(p)
print(geo.which_entity(q2))

op_type = get_op_type(op)
print(op_type)


s1: 6.5 e123∧einf - e123∧eo - 3 e12∧einf∧eo + 2 e13∧einf∧eo - e23∧einf∧eo

op_d: - e1 - 2 e2 - 3 e3 - 6.5 einf - eo

Point(0.93, 1.86, 2.79)
Point(0.93, 1.86, 2.79)
Versor grade: 4


op: - e1 - 2 e2 - 3 e3 - 6.5 einf - eo

New versor grade: 1
('inversion', -e1 - 2 e2 - 3 e3 - 6.5 einf - eo)


In [87]:
# Inversion in a sphere at a point
s1 = geo.create(Sphere(Point(0,0,0), 1))
s1.show("s1")
s1_d = s1.dual()
s1_d.show("s1_d")

p = geo.create(Point(2,0,0))
q1 = s1.vp(p)
print(geo.which_entity(q1))
q2 = s1_d.vp(p)
print(geo.which_entity(q2))

op_type = get_op_type(op)
print(op_type)


s1: - 0.5 e123∧einf - e123∧eo

s1_d: 0.5 einf - eo

Point(0.50, 0.00, 0.00)
Point(0.50, -0.00, -0.00)
Versor grade: 4


op: - e1 - 2 e2 - 3 e3 - 6.5 einf - eo

New versor grade: 1
('inversion', -e1 - 2 e2 - 3 e3 - 6.5 einf - eo)


In [88]:
# Reflection in a point
s1 = geo.create(Sphere(Point(1,2,3), 1))
op = s1.dual() ^ N3.einf
op.show("op")
op_d = op.dual()
op_d.show("op_d")

p = geo.create(Point(0,0,0))
q1 = op.vp(p)
print(geo.which_entity(q1))
q2 = op_d.vp(p)
print(geo.which_entity(q2))

op_type = get_op_type(op)
print(op_type)

op: - e1∧einf - 2 e2∧einf - 3 e3∧einf + einf∧eo

op_d: - e123 - 3 e12∧einf + 2 e13∧einf - e23∧einf

Point(2.00, 4.00, 6.00)
Point(2.00, 4.00, 6.00)
Versor grade: 2


op: - e1∧einf - 2 e2∧einf - 3 e3∧einf + einf∧eo

New versor grade: 2
('ref_in_point', -e1∧einf - 2 e2∧einf - 3 e3∧einf + einf∧eo)


In [89]:
# Reflection in a line
s1 = geo.create(Sphere(Point(1,1,0), 1))
s2 = geo.create(Sphere(Point(2,2,1), 1))
op = s1.dual() ^ s2.dual() ^ N3.einf
op.show("op")
op_d = op.dual()
op_d.show("op_d")

p = geo.create(Point(0,0,0))
q1 = op.vp(p)
print(geo.which_entity(q1))
q2 = op_d.vp(p)
print(geo.which_entity(q2))

op_type = get_op_type(op)
print(op_type)

op: e13∧einf + e1∧einf∧eo + e23∧einf + e2∧einf∧eo + e3∧einf∧eo

op_d: - e12 + e13 - e1∧einf - e23 + e2∧einf

Point(0.67, 0.67, -1.33)
Point(0.67, 0.67, -1.33)
Versor grade: 3


op: - e12 + e13 - e1∧einf - e23 + e2∧einf

New versor grade: 2
('ref_in_line', -e12 + e13 - e1∧einf - e23 + e2∧einf)


In [90]:
# Reflection in a line through origin
s1 = geo.create(Sphere(Point(1,1,0), 1))
s2 = geo.create(Sphere(Point(0,0,0), 1))
op = s1.dual() ^ s2.dual() ^ N3.einf
op.show("op")
op_d = op.dual()
op_d.show("op_d")

p = geo.create(Point(1,2,3))
q1 = op.vp(p)
print(geo.which_entity(q1))
q2 = op_d.vp(p)
print(geo.which_entity(q2))

op_type = get_op_type(op)
print(op_type)

op: - e1∧einf∧eo - e2∧einf∧eo

op_d: - e13 + e23

Point(2.00, 1.00, -3.00)
Point(2.00, 1.00, -3.00)
Versor grade: 3


op: - e13 + e23

New versor grade: 2
('ref_in_line', -e13 + e23)


In [91]:
# Reflection in a plane
s1 = geo.create(Sphere(Point(1,1,0), 1))
s2 = geo.create(Sphere(Point(0,1,0), 1))
s3 = geo.create(Sphere(Point(0,0,1), 1))
op = s1.dual() ^ s2.dual() ^ s3.dual() ^ N3.einf
op.show("op")
op_d = op.dual()
op_d.show("op_d")

p = geo.create(Point(0,0,0))
q1 = op.vp(p)
print(geo.which_entity(q1))
q2 = op_d.vp(p)
print(geo.which_entity(q2))

op_type = get_op_type(op)
print(op_type)

op: - e123∧einf + e12∧einf∧eo - e13∧einf∧eo

op_d: e2 + e3 + einf

Point(0.00, 1.00, 1.00)
Point(-0.00, 1.00, 1.00)
Versor grade: 4


op: e2 + e3 + einf

New versor grade: 1
('reflection plane', e2 + e3 + einf)


In [92]:
# Reflection in a plane through origin
s1 = geo.create(Sphere(Point(1,1,0), 1))
s2 = geo.create(Sphere(Point(0,1,0), 1))
s3 = geo.create(Sphere(Point(0,0,0), 1))
op = s1.dual() ^ s2.dual() ^ s3.dual() ^ N3.einf
op.show("op")
op_d = op.dual()
op_d.show("op_d")

p = geo.create(Point(0,0,0))
q1 = op.vp(p)
print(geo.which_entity(q1))
q2 = op_d.vp(p)
print(geo.which_entity(q2))

op_type = get_op_type(op)
print(op_type)

op: e12∧einf∧eo

op_d: e3

Point(0.00, 0.00, 0.00)
Point(-0.00, -0.00, -0.00)
Versor grade: 4


op: e3

New versor grade: 1
('reflection plane', e3)
